# Logic-Zero · Task 7b: Re-evaluate SFT checkpoint

Standalone re-evaluation of the saved best SFT checkpoint with **fixed parameters**:
- `max_new_tokens=1000` (was 700 — truncated 17-27% of n=4-6 generations)
- Per-n accuracy breakdown
- Frequent progress prints (every 5 puzzles)
- Strict GPU asserts (no silent CPU fallback)
- Auto-restore checkpoint from Drive

**Runtime:** GPU (T4 or L4). **Duration:** ~15-25 min.

**Before running:** Runtime → Restart runtime (clears any GPU memory from prior training).

## 1. Verify GPU + clean state

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')
    assert free / 1e9 > 5, 'Less than 5GB free — restart runtime first (Runtime → Restart runtime)'
else:
    raise RuntimeError('NO GPU — switch runtime type to GPU')
print(f'torch {torch.__version__}  bf16: {torch.cuda.is_bf16_supported()}')

## 2. Pull latest repo code

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

## 3. Install dependencies

Same as 01_sft_train — keeps Colab's torch + adds only what we need.

In [ ]:
import subprocess
def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(['pip', 'install', '-q', '--default-timeout=180'] + specs,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed: {specs}')
pip_inst(['trl>=0.13.0,<0.15.0', 'peft>=0.13.0,<0.16.0', 'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv'])  # train.common imports z3 at module top
import transformers, peft, z3
print(f'transformers={transformers.__version__}  peft={peft.__version__}  z3={z3.get_version_string()}')

## 4. Mount Drive + restore checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, json

drive_ckpt = '/content/drive/MyDrive/logic-zero/checkpoints/sft/best'
local_ckpt = '/content/logic-zero/results/checkpoints/sft/best'

# Restore from Drive if local missing
if not os.path.isdir(local_ckpt):
    assert os.path.isdir(drive_ckpt), f'NO CHECKPOINT in Drive: {drive_ckpt}'
    os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
    shutil.copytree(drive_ckpt, local_ckpt)
    print(f'✓ Restored from Drive → {local_ckpt}')
else:
    print(f'✓ Local checkpoint already present: {local_ckpt}')

# Verify integrity
adapter = f'{local_ckpt}/adapter_model.safetensors'
assert os.path.exists(adapter), f'MISSING: {adapter}'
size_mb = os.path.getsize(adapter) / 1e6
assert size_mb > 10, f'Adapter too small ({size_mb}MB) — likely corrupted'
meta = json.load(open(f'{local_ckpt}/dev_acc.json'))
print(f'\nCheckpoint: {size_mb:.1f}MB')
print(f'Original dev_acc (max_new_tokens=700): {meta["acc"]:.3f} at epoch {meta["epoch"]}')
print('Files:', os.listdir(local_ckpt))

## 5. Load model + LoRA adapter (with strict GPU check)

In [ ]:
import time
from peft import PeftModel
from train.common import load_base_model

print('[1/2] Loading base Qwen2.5-1.5B...', flush=True)
t0 = time.time()
model, tok = load_base_model()
# load_base_model() returns model on CPU by default — must move to GPU
# explicitly (SFTTrainer does this internally during training, but eval
# code has to do it itself).
model = model.to('cuda')
print(f'  loaded in {time.time()-t0:.0f}s', flush=True)

print('[2/2] Loading LoRA adapter...', flush=True)
t0 = time.time()
model = PeftModel.from_pretrained(model, local_ckpt)
model.eval()
model.config.use_cache = True  # KV cache for ~10x faster generation
print(f'  loaded in {time.time()-t0:.0f}s', flush=True)

# Strict device check — no silent CPU fallback
dev = next(model.parameters()).device
assert dev.type == 'cuda', f'MODEL ON {dev} — generation will be 100x slower. Restart runtime!'
print(f'\n✓ Model on {dev}')
print(f'  GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f}GB')

## 6. Run dev eval with `max_new_tokens=1000`

170 puzzles. Progress every 5. ETA ~15-25 min on T4, ~10-15 min on L4.

In [ ]:
from train.common import to_chat, extract_answer

dev_data = [json.loads(l) for l in open('data/dev_data.jsonl')]
print(f'Dev set: {len(dev_data)} puzzles')
from collections import Counter
print(f'Per-n distribution: {dict(sorted(Counter(len(r["ground_truth"]) for r in dev_data).items()))}\n')

per_n_correct, per_n_total = {}, {}
wrong_examples = []  # save a few wrongs for inspection
t0 = time.time()

with torch.no_grad():
    for i, rec in enumerate(dev_data):
        n = len(rec['ground_truth'])
        per_n_total[n] = per_n_total.get(n, 0) + 1
        inputs = tok(to_chat(tok, rec['puzzle']), return_tensors='pt').to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=1000, do_sample=False,
            pad_token_id=tok.eos_token_id,
            stop_strings=['</answer>'], tokenizer=tok,
            temperature=None, top_p=None, top_k=None,
        )
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = extract_answer(resp, n=n)
        is_correct = pred == rec['ground_truth']
        if is_correct:
            per_n_correct[n] = per_n_correct.get(n, 0) + 1
        elif len(wrong_examples) < 5 and n in (5, 6):
            wrong_examples.append({'n': n, 'gt': rec['ground_truth'], 'pred': pred, 'resp_tail': resp[-300:]})
        if (i+1) % 5 == 0:
            tot = sum(per_n_correct.values())
            eta = (time.time()-t0) / (i+1) * (len(dev_data) - i - 1)
            print(f'  {i+1:3d}/{len(dev_data)}  n={n}  acc={tot/(i+1):.3f}  '
                  f'elapsed={time.time()-t0:.0f}s  eta={eta:.0f}s', flush=True)

print(f'\n=== Done in {time.time()-t0:.0f}s ===')

## 7. Results: per-n breakdown + diagnosis

In [ ]:
print('=== Per-n accuracy (max_new_tokens=1000) ===')
for n in sorted(per_n_total):
    c, t = per_n_correct.get(n, 0), per_n_total[n]
    bar = '█' * int(c/t * 30)
    print(f'  n={n}: {c:3d}/{t:3d} = {c/t:.1%}  {bar}')

total_c, total_t = sum(per_n_correct.values()), sum(per_n_total.values())
new_acc = total_c / total_t
print(f'\nNEW dev_acc = {total_c}/{total_t} = {new_acc:.1%}')
print(f'OLD dev_acc (max_new_tokens=700) = {meta["acc"]:.1%}')
print(f'Improvement from fixing truncation: {(new_acc - meta["acc"])*100:+.1f}pp')

# Sanity criteria (notebook 01_sft_train spec)
print('\n=== Decision ===')
if new_acc >= 0.50:
    print('✅ dev_acc ≥ 50%: pipeline healthy. Proceed to DPO.')
elif new_acc >= 0.25:
    print('✅ dev_acc ≥ 25%: proceed to DPO, expect +5-10pp gain.')
else:
    print('⚠️ dev_acc < 25%: borderline. Check n=2/3 — if those ≥40%, DPO can still help on easy puzzles.')

## 8. Save results + sample errors

In [ ]:
result = {
    'dev_acc': new_acc,
    'old_dev_acc': meta['acc'],
    'best_epoch': meta['epoch'],
    'per_n': {str(n): {'correct': per_n_correct.get(n, 0), 'total': per_n_total[n],
                       'acc': per_n_correct.get(n, 0) / per_n_total[n]}
              for n in sorted(per_n_total)},
    'max_new_tokens': 1000,
    'wrong_samples_n5_6': wrong_examples,
}

os.makedirs('results', exist_ok=True)
with open('results/sft_dev_reeval.json', 'w') as f:
    json.dump(result, f, indent=2)
print('✓ Saved results/sft_dev_reeval.json')

# Backup to Drive
shutil.copy('results/sft_dev_reeval.json',
            '/content/drive/MyDrive/logic-zero/checkpoints/sft/sft_dev_reeval.json')
print('✓ Backed up to Drive')

# Inspect a few n=5/6 wrongs
if wrong_examples:
    print('\n=== Sample n=5/6 failures ===')
    for w in wrong_examples[:3]:
        print(f'\nn={w["n"]}  GT={w["gt"]}  PRED={w["pred"]}')
        print(f'  tail: ...{w["resp_tail"]}')

## ✅ Re-eval done

**Output:** `results/sft_dev_reeval.json` (local + Drive backup)

### If results look surprising (n=2 < 70% or n=3 < 35%) → run section 9

Section 9 classifies failures as parser-bug / real-reasoning / format-error so you know whether to fix the parser or accept the result.

### Otherwise: next steps based on dev_acc

| new dev_acc | Action |
|---|---|
| ≥ 25% | → Task 8: full eval on 1530-puzzle eval set, then Task 9: DPO |
| 15-25% | → Still proceed to DPO; expect gain mostly on n=2/3/4 |
| < 15% | → Run section 9 diagnostic first |

## 9. Diagnostic: classify n=2/n=3 failures

Run **AFTER** sections 6+7 finish. Re-runs inference on 10 n=2 + 10 n=3 puzzles, saves full responses, and classifies each failure as:

- **PARSER_BUG** — strict `extract_answer` missed it, but a loose parser finds the correct answer in the response
- **REASONING_ERROR** — both parsers got an answer, but it's wrong
- **FORMAT_ERROR** — neither parser can extract anything

Decision:
- PARSER_BUG ≥ 30% → fix `_PAIR_RE` regex, re-eval — likely gain +5-15pp
- REASONING_ERROR > 60% → real ceiling, but DPO can still help
- FORMAT_ERROR > 30% → training data format inconsistency

Time: ~8-10 min on T4 (20 puzzles × ~25s each).

In [ ]:
import re, time, torch, json, os
from collections import Counter
from train.common import to_chat, extract_answer

# Sanity: model + tok must already be loaded from section 5
assert 'model' in dir() and 'tok' in dir(), 'Run section 5 (load model) first'
assert next(model.parameters()).device.type == 'cuda', 'Model not on GPU'

def loose_extract(response, n):
    """Permissive parser - tries many formats the strict one misses.
    Returns dict {letter: identity} or None."""
    text = response[-500:]  # focus on answer area
    expected = [chr(ord('A')+i) for i in range(n)]
    result = {}
    patterns = [
        # Various separators between letter and identity
        r'\b{0}\s*[:=\-—→.)]+\s*(?:a\s+|an\s+|the\s+)?(knight|knave)',
        # "X is (a/the) knight/knave"
        r'\b{0}\s+is\s+(?:a\s+|an\s+|the\s+)?(knight|knave)',
        # "X must/should/will be (a) knight"
        r'\b{0}\s+(?:must|should|will|would|can|could)\s+be\s+(?:a\s+|an\s+|the\s+)?(knight|knave)',
        # Catch-all: letter then knight/knave within 25 non-letter chars
        r'\b{0}\b[^A-Za-z\n]{{0,25}}(knight|knave)',
    ]
    for letter in expected:
        identity = None
        for pat in patterns:
            matches = list(re.finditer(pat.format(letter), text, re.IGNORECASE))
            if matches:
                identity = matches[-1].group(1).lower()  # last match = final answer
                break
        if identity is None:
            return None
        result[letter] = identity
    return result if len(result) == n else None

# Subset: 10 n=2 + first 10 n=3 puzzles from dev set
n2_samples = [r for r in dev_data if len(r['ground_truth']) == 2][:10]
n3_samples = [r for r in dev_data if len(r['ground_truth']) == 3][:10]
subset = n2_samples + n3_samples
print(f'Running diagnostic on {len(subset)} puzzles ({len(n2_samples)}x n=2 + {len(n3_samples)}x n=3)...', flush=True)
print(f'Expected duration: ~{len(subset) * 25}s\n', flush=True)

diag_results = []
t0 = time.time()
with torch.no_grad():
    for i, rec in enumerate(subset):
        n = len(rec['ground_truth'])
        inputs = tok(to_chat(tok, rec['puzzle']), return_tensors='pt').to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=1000, do_sample=False,
            pad_token_id=tok.eos_token_id,
            stop_strings=['</answer>'], tokenizer=tok,
            temperature=None, top_p=None, top_k=None,
        )
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        strict = extract_answer(resp, n=n)
        loose = loose_extract(resp, n=n)
        gt = rec['ground_truth']
        if strict == gt:
            cat = 'CORRECT'
        elif loose == gt:
            cat = 'PARSER_BUG'
        elif loose is None:
            cat = 'FORMAT_ERROR'
        else:
            cat = 'REASONING_ERROR'
        diag_results.append({'n': n, 'gt': gt, 'strict': strict, 'loose': loose, 'resp': resp, 'cat': cat})
        print(f'  {i+1:2d}/{len(subset)}  n={n}  {cat:16s}  ({time.time()-t0:.0f}s)', flush=True)

# === Summary ===
sep = '=' * 55
print(f'\n{sep}\nDiagnostic Summary ({time.time()-t0:.0f}s)\n{sep}')
for n in [2, 3]:
    by_n = [r for r in diag_results if r['n'] == n]
    cats = Counter(r['cat'] for r in by_n)
    total = len(by_n)
    print(f'\nn={n} ({total} puzzles):')
    for cat in ['CORRECT', 'PARSER_BUG', 'FORMAT_ERROR', 'REASONING_ERROR']:
        cnt = cats.get(cat, 0)
        pct = 100 * cnt / total
        bar = '#' * int(pct / 5)
        print(f'  {cat:16s} {cnt:2d}/{total} = {pct:5.1f}%  {bar}')

# Combined verdict
all_cats = Counter(r['cat'] for r in diag_results)
total_all = len(diag_results)
parser_pct = 100 * all_cats.get('PARSER_BUG', 0) / total_all
reasoning_pct = 100 * all_cats.get('REASONING_ERROR', 0) / total_all
format_pct = 100 * all_cats.get('FORMAT_ERROR', 0) / total_all
correct_pct = 100 * all_cats.get('CORRECT', 0) / total_all

print(f'\n{sep}\nVERDICT\n{sep}')
print(f'Strict-parser acc: {correct_pct:.0f}%')
print(f'If parser fixed:    {correct_pct + parser_pct:.0f}% (+{parser_pct:.0f}pp)')
print()
if parser_pct >= 30:
    print(f'>>> PARSER BUG CONFIRMED ({parser_pct:.0f}% of cases).')
    print(f'    Action: relax _PAIR_RE in train/common.py to match more formats,')
    print(f'    then re-run section 6. Expected gain: +{parser_pct:.0f}pp.')
elif reasoning_pct >= 50:
    print(f'>>> REAL REASONING WEAKNESS ({reasoning_pct:.0f}% wrong reasoning).')
    print(f'    The 1.5B model with current SFT data is hitting its ceiling.')
    print(f'    Action: proceed to DPO anyway -- it is designed for exactly this.')
    print(f'    Or: scale up to Qwen2.5-3B / expand SFT data.')
elif format_pct >= 30:
    print(f'>>> FORMAT DRIFT ({format_pct:.0f}% no recoverable answer).')
    print(f'    Model did not learn the <answer> format reliably.')
    print(f'    Action: inspect training data format consistency.')
else:
    print(f'Mixed signal. Inspect raw responses below.')

# === Detailed view of every non-CORRECT case ===
print(f'\n{sep}\nDETAILED FAILURES\n{sep}')
for i, r in enumerate(diag_results):
    if r['cat'] == 'CORRECT':
        continue
    print(f'\n--- #{i+1}  n={r["n"]}  {r["cat"]} ---')
    print(f'GT:     {r["gt"]}')
    print(f'STRICT: {r["strict"]}')
    print(f'LOOSE:  {r["loose"]}')
    print(f'RESPONSE TAIL (last 400 chars):')
    print(r['resp'][-400:])

# Save for later analysis
os.makedirs('results', exist_ok=True)
save_data = [{k: v for k, v in r.items() if k != 'resp'} | {'resp_tail': r['resp'][-600:]}
             for r in diag_results]
with open('results/sft_diagnostic.json', 'w') as f:
    json.dump({'summary': dict(all_cats), 'cases': save_data}, f, indent=2)
import shutil
shutil.copy('results/sft_diagnostic.json',
            '/content/drive/MyDrive/logic-zero/checkpoints/sft/sft_diagnostic.json')
print(f'\nSaved results/sft_diagnostic.json (+ Drive backup)')
